1. Libraries


In [1]:
import os
import langchain_core
from langchain_core.messages import SystemMessage, ToolMessage,HumanMessage, AIMessage
import langchain_core.prompts
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

2. Initialize an LLM 

In [2]:
llm = init_chat_model(model_provider='google_genai', model='gemini-3.5-flash-lite', temperature=0.9, max_output_tokens=1024)

3. Load environmental keys and prep

In [3]:
from dotenv import load_dotenv
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [4]:
company_name_user = input("Enter the company name for due diligence snapshot: ")

TASK = (
    f"Compile a due-diligence snapshot for vendor '{company_name_user}'. "
    "I need: founding year, latest funding round "
    "(amount + date + lead investor), CEO name, reported ARR if available, "
    "and any legal or compliance red flags. Include today's date in the snapshot."
)
print("Task:", TASK)

Task: Compile a due-diligence snapshot for vendor 'cipla'. I need: founding year, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and any legal or compliance red flags. Include today's date in the snapshot.


In [5]:
import os
import requests

def get_tavily_results(query, include_domains=None, search_depth="basic", max_results=2):
    response = requests.post(
    "https://api.tavily.com/search",
    headers={"Content-Type": "application/json"},
    json={
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": query,
        "include_domains": include_domains or ["https://www.screener.in/"],  # specific website
        "search_depth": search_depth,
        "max_results": max_results,
    },
    )

    response.raise_for_status()

    return response

In [6]:
def show_tavily_results(response):
    for result in response.json()["results"]:
        print(result["title"])
        print(result["url"])
        print(result["content"])
        print(result)

In [7]:
response = get_tavily_results("Cipla company profile", include_domains=["https://www.screener.in/"], search_depth="basic", max_results=7)

show_tavily_results(response)

Cipla Ltd share price | About Cipla | Key Insights - Screener
https://www.screener.in/company/CIPLA/consolidated
# Cipla Ltd

AI

Cipla Summary  Chart Analysis Peers Quarters Profit & Loss Balance Sheet Cash Flow Ratios Investors Documents

AI

# Cipla Ltd

₹ 1,389  -0.59%

28 Sep 3:29 p.m.

cipla.com   BSE: 500087   NSE: CIPLA   F&O

About

Incorporated in 1935, Cipla Ltd is in the business of manufacturing, developing, and marketing a wide range of branded and generic formulations and APIs(

Key Points [...] | India IPM Brands with Revenue > INR 100 Cr count |
| India Trade Generics Stockist Network count |
| South Africa Overall Market Share % |
| US Albuterol Total MDI Market Share % |
| India IPM Top 300 Brand Count count |
| SA Private Market Growth (Cipla YoY, ZAR terms) % |
| India Chronic Therapy Mix % |
| R&D Expenditure (Percentage of Turnover) % ・Standalone data | [...] - The Operations and Administrative Committee of the Company on 24th September, 2026 has allotted 12,168 

In [11]:
@tool
def get_company_profile(company_name: str) -> str:
    """Return basic company profile: founding year, HQ, sector, key products."""
    tavily_response = get_tavily_results(
        f"{company_name} company profile",
        include_domains=["screener.in"],
        search_depth="basic",
        max_results=7,
    )

    data = tavily_response.json()
    results = data.get("results", [])

    if not results:
        return "No results found."

    return (
        "Company profile retrieved from Tavily API.\n\n"
        + "\n\n".join(
            f"Title: {item.get('title', '')}\n"
            f"URL: {item.get('url', '')}\n"
            f"{item.get('content', '')}"
            for item in results
        )
    )

In [12]:
print(get_company_profile.invoke({"company_name": company_name_user}))

Company profile retrieved from Tavily API.

Title: Cipla Ltd share price | About Cipla | Key Insights - Screener
URL: https://www.screener.in/company/CIPLA/consolidated
# Cipla Ltd

AI

Cipla Summary  Chart Analysis Peers Quarters Profit & Loss Balance Sheet Cash Flow Ratios Investors Documents

AI

# Cipla Ltd

₹ 1,389  -0.59%

28 Sep 3:29 p.m.

cipla.com   BSE: 500087   NSE: CIPLA   F&O

About

Incorporated in 1935, Cipla Ltd is in the business of manufacturing, developing, and marketing a wide range of branded and generic formulations and APIs(

Key Points [...] Key Points

Geographical Split FY26:(  
 One India - 45%  
 North America - 24%  
 One Africa - 15%  
 EMEU - 13%  
 Others - 3%

Website   BSE   NSE   F&O

 Market Cap   ₹ 1,12,205 Cr.
 Current Price   ₹ 1,389
 High / Low  ₹ 1,673 / 1,166
 Stock P/E   31.3
 Book Value   ₹ 426
 Dividend Yield   0.93 %
 ROCE   15.5 %
 ROE   11.6 %
 Face Value   ₹ 2.00

Edit ratios

Pros

 Company is almost debt free.
 Company has been maintai

In [13]:
response = get_tavily_results("Cipla funding rounds", include_domains=["https://www.crunchbase.com"], search_depth="basic", max_results=2)
show_tavily_results(response)


advanced = get_tavily_results("cipla funding rounds", include_domains=[response.json()["results"][0]["url"]], search_depth="advanced", max_results=3)
show_tavily_results(advanced) 

Cipla - Crunchbase Company Profile & Funding
https://www.crunchbase.com/organization/cipla
Cipla closed its last funding round on obfuscated obfuscation from a Post-IPO Secondary round.
{'url': 'https://www.crunchbase.com/organization/cipla', 'title': 'Cipla - Crunchbase Company Profile & Funding', 'content': 'Cipla closed its last funding round on obfuscated obfuscation from a Post-IPO Secondary round.', 'score': 0.7975099, 'raw_content': None, 'id': '506f4a-00'}
Cipla Health - Crunchbase Company Profile & Funding
https://www.crunchbase.com/organization/cipla-health
Unlock company funding data, including rounds, dates, amounts, and investors. obfuscated. Register for Free
{'url': 'https://www.crunchbase.com/organization/cipla-health', 'title': 'Cipla Health - Crunchbase Company Profile & Funding', 'content': 'Unlock company funding data, including rounds, dates, amounts, and investors. obfuscated. Register for Free', 'score': 0.5198388, 'raw_content': None, 'id': '59cb00-01'}
Cipla - 

In [14]:
@tool
def get_funding_rounds(company_name: str) -> str:
    """Return a list of funding rounds for the company, including amount, date, and lead investor."""
    if "arion" in company_name.lower():
        return (
            "Arion Data Systems Funding Rounds:\n"
            "1. Seed Round: $2M, Jan 2018, Lead Investor: TechVentures\n"
            "2. Series A: $10M, Mar 2020, Lead Investor: GrowthCapital\n"
            "3. Series B: $25M, Jul 2022, Lead Investor: FutureFund"
        )
    else:
        response = get_tavily_results("Cipla funding rounds", include_domains=["https://www.crunchbase.com"], search_depth="basic", max_results=1)
        advanced = get_tavily_results("cipla funding rounds", include_domains=[response.json()["results"][0]["url"]], search_depth="advanced", max_results=3)
        string_result = ""
        for result in advanced.json()["results"]:
            print(result)
            string_result += 'title:' + str(result["title"]) + ', url:' + str(result["url"]) + ', content:' + str(result["content"])
            string_result += '\n\n'    
        return "Results: "+ string_result if advanced.json()["results"] else "No funding round information found."

In [15]:
response =get_funding_rounds.invoke({"company_name": "Cipla"})
print(response)

{'url': 'https://www.crunchbase.com/organization/cipla', 'title': 'Cipla - Crunchbase Company Profile & Funding', 'content': 'Cipla is a drug manufacturing company that focuses on a variety of pharmaceutical products. Its focal areas are biologicals, small-molecule innovation,', 'score': 0.40967003, 'raw_content': None, 'id': '3ea127-00'}
{'url': 'https://www.crunchbase.com/organization/cipla/profiles_and_contacts', 'title': 'Profiles & Contacts - Cipla', 'content': 'Cipla is a drug manufacturing company that focuses on a variety of pharmaceutical products.', 'score': 0.12217739, 'raw_content': None, 'id': '92425f-01'}
Results: title:Cipla - Crunchbase Company Profile & Funding, url:https://www.crunchbase.com/organization/cipla, content:Cipla is a drug manufacturing company that focuses on a variety of pharmaceutical products. Its focal areas are biologicals, small-molecule innovation,

title:Profiles & Contacts - Cipla, url:https://www.crunchbase.com/organization/cipla/profiles_and_co

In [16]:

@tool
def search_news(company_name: str, topic: str) -> str:
    """Search recent news about a company on a specific topic such as 'CEO and leadership',
    'revenue and ARR', 'GDPR legal risk', or 'customer contracts'. One call per topic."""
    key = f"{company_name} {topic}".lower()
    if "arion" in key:
        if any(t in key for t in ("ceo", "leadership", "executive", "founder")):
            return (
                "CEO: Jonas Meier (co-founder, formerly Palantir). "
                "CTO: Priya Natarajan (joined 2021 from AWS). No recent changes."
            )
        if any(t in key for t in ("revenue", "arr", "financial", "growth")):
            return (
                "Reported ARR EUR 12M (Q4 2023 investor deck). NRR 118%. "
                "No audited public accounts (private company)."
            )
        if any(t in key for t in ("legal", "gdpr", "compliance", "risk", "red flag")):
            return (
                "Open GDPR complaint with Dutch DPA (Feb 2023). "
                "IP dispute with former engineer settled 2022. No other litigation."
            )
        if any(t in key for t in ("customer", "client", "partner")):
            return (
                "Customers: ING Bank (NL), Medicover (PL), unnamed UK insurer. "
                "Microsoft Azure partner. SOC 2 Type II certified."
            )
        return "No significant news for that topic."
    else:
        response = get_tavily_results(f"{company_name} {topic}", include_domains=["https://www.cnbc.com"], search_depth="basic", max_results=3)
                
        data = response.json()

        if not data.get("results"):
            return "No news found."

        return "\n\n".join(
            f"{result.get('title', '')}\n"
            f"{result.get('url', '')}\n"
            f"{result.get('content', '')}"
            for result in data["results"]
        )

from datetime import date
@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return str(date.today())

In [17]:
print(search_news.invoke({"company_name": "aeroflex", "topic": "CEO and leadership"}))

AEROFLEX-IN: Aeroflex Industries Ltd - Stock Price, Quote and News - CNBC
https://www.cnbc.com/quotes/AEROFLEX-IN
Asad Daud

Executive Chairman of the Board, Managing Director [...] Aeroflex Industries Limited is an India-based manufacturer of metallic flexible flow solutions made with stainless steel. The Company facilitates the controlled flow of gas, liquid, and solid substances across various industries, and caters to global as well as domestic markets. The Company is engaged in manufacturing stainless steel, flexible hose with braiding and without braiding and assemblies. Its products include Corrugated Stainless-Steel Hose, Double Interlock...

Asad Daud

AEROENTER-IN: Aeroflex Enterprises Ltd - Stock Price, Quote and News - CNBC
https://www.cnbc.com/quotes/AEROENTER-IN
Aeroflex Enterprises Limited, formerly SAT Industries Limited, is an India-based company, which is engaged in the manufacturing of flexible packaging, hose pipes, leasing, investment and other activities. Its segm

In [18]:
print(get_current_date.invoke({input: {}}))

2026-11-11


In [19]:
TOOLS = [get_company_profile, get_funding_rounds, search_news, get_current_date]

# Inspect what @tool generated
for t in TOOLS:
    print(f"Tool: {t.name:30} | Args: {list(t.args.keys())} | Description: {t.description} | Content: {t.func} ")


Tool: get_company_profile            | Args: ['company_name'] | Description: Return basic company profile: founding year, HQ, sector, key products. | Content: <function get_company_profile at 0x000002722C1BEC40> 
Tool: get_funding_rounds             | Args: ['company_name'] | Description: Return a list of funding rounds for the company, including amount, date, and lead investor. | Content: <function get_funding_rounds at 0x000002722C2271C0> 
Tool: search_news                    | Args: ['company_name', 'topic'] | Description: Search recent news about a company on a specific topic such as 'CEO and leadership',
'revenue and ARR', 'GDPR legal risk', or 'customer contracts'. One call per topic. | Content: <function search_news at 0x000002722C2273D0> 
Tool: get_current_date               | Args: [] | Description: Return today's date in YYYY-MM-DD format. | Content: <function get_current_date at 0x000002722C2275E0> 


6. Tools Map

In [20]:
llm_with_tools = llm.bind_tools(TOOLS)

SYSTEM_PROMPT = (
    "You are a corporate research assistant conducting vendor due diligence. "
    "Use the available tools to gather factual information. "
    "When you have enough information, produce a structured due-diligence snapshot."
)

TOOL_MAP = {t.name: t for t in TOOLS}
print("Tools bound:", [t.name for t in TOOLS])


Tools bound: ['get_company_profile', 'get_funding_rounds', 'search_news', 'get_current_date']


7.States

In [21]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=TASK),
]

print(f"Initial messages: {len(messages)}")
for m in messages:
    print(f"  [{m.__class__.__name__}]: {str(m.content)[:70]}")


Initial messages: 2
  [SystemMessage]: You are a corporate research assistant conducting vendor due diligence
  [HumanMessage]: Compile a due-diligence snapshot for vendor 'cipla'. I need: founding 


In [22]:
print(messages[0].type + ':' + str(messages[0].content))
print(messages[1].__class__.__name__ + ':' + str(messages[1].content))

system:You are a corporate research assistant conducting vendor due diligence. Use the available tools to gather factual information. When you have enough information, produce a structured due-diligence snapshot.
HumanMessage:Compile a due-diligence snapshot for vendor 'cipla'. I need: founding year, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and any legal or compliance red flags. Include today's date in the snapshot.


In [23]:
print("Starting agent loop...")

Starting agent loop...


8. Looping the agent

In [24]:
MAX_STEPS = 10
step = 0
final_response = None

print("=" * 60)
print(f"GOAL: {TASK}")
print("=" * 60)

while step < MAX_STEPS:
    step += 1
    print(f"\n[Step {step}] Calling model...")

    # -- Reason ----------------------------------------------------------
    response = llm_with_tools.invoke(messages)
    messages.append(response)   # AIMessage (may contain tool_calls)

    # -- Decide ----------------------------------------------------------
    if not response.tool_calls:
        print(f"[Step {step}] Final answer received.")
        final_response = response
        break

    print(f"[Step {step}] Tool calls: {[tc['name'] for tc in response.tool_calls]}")

    # -- Act + Update ---------------------------------------------------
    for tc in response.tool_calls:
        tool_fn = TOOL_MAP[tc["name"]]
        result  = tool_fn.invoke(tc["args"])
        print(f"  {tc['name']}({tc['args']}) => {str(result)[:80]!r}...")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print(response.content)
print("=" * 60)


GOAL: Compile a due-diligence snapshot for vendor 'cipla'. I need: founding year, latest funding round (amount + date + lead investor), CEO name, reported ARR if available, and any legal or compliance red flags. Include today's date in the snapshot.

[Step 1] Calling model...


c:\Users\jayar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[Step 1] Tool calls: ['get_current_date', 'get_company_profile', 'get_funding_rounds', 'search_news', 'search_news', 'search_news']
  get_current_date({}) => '2026-11-11'...
  get_company_profile({'company_name': 'Cipla'}) => 'Company profile retrieved from Tavily API.\n\nTitle: Cipla Ltd share price | About'...
{'url': 'https://www.crunchbase.com/organization/cipla', 'title': 'Cipla - Crunchbase Company Profile & Funding', 'content': 'Cipla is a drug manufacturing company that focuses on a variety of pharmaceutical products. Its focal areas are biologicals, small-molecule innovation,', 'score': 0.40995342, 'raw_content': None, 'id': 'bc75c0-00'}
{'url': 'https://www.crunchbase.com/organization/cipla/profiles_and_contacts', 'title': 'Profiles & Contacts - Cipla', 'content': 'Cipla is a drug manufacturing company that focuses on a variety of pharmaceutical products. Mumbai, Maharashtra, India 10001+ . Cipla has 8 board members and', 'score': 0.10071066, 'raw_content': None, 'id': '10598

c:\Users\jayar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[Step 2] Final answer received.

FINAL ANSWER:
[{'type': 'text', 'text': '### Due-Diligence Snapshot: Cipla Ltd.\n\n* **Date of Report:** November 11, 2026\n* **Company Name:** Cipla Ltd.\n* **Founding Year:** 1935\n* **Latest Funding Round:** Not applicable. Cipla is a publicly traded multinational pharmaceutical company listed on the Bombay Stock Exchange (BSE: 500087) and the National Stock Exchange of India (NSE: CIPLA), rather than a privately backed startup raising venture capital.\n* **Chief Executive Officer (CEO):** Achin Gupta (Global Chief Executive Officer and Managing Director)\n* **Reported ARR (Annual Recurring Revenue):** Not applicable. As a traditional pharmaceutical manufacturing and commercialization company, Cipla reports annual revenue and financial results via public quarterly/annual reports rather than SaaS-style ARR metrics. (For context, the company maintains a multi-billion-dollar top-line footprint globally across branded generics, APIs, and formulations).\n

In [27]:
print(f"Total messages: {len(messages)}  |  Steps: {step}")
print()
for i, msg in enumerate(messages):
    cls = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
        print(f"[{i}] {cls}: tool_calls = {calls}")
    elif hasattr(msg, "tool_call_id"):
        print(f"[{i}] {cls}: id={msg.tool_call_id} | {str(msg.content)[:80]}")
    else:
        print(f"[{i}] {cls}: {str(msg.content)[:100]}")


Total messages: 10  |  Steps: 2

[0] SystemMessage: You are a corporate research assistant conducting vendor due diligence. Use the available tools to g
[1] HumanMessage: Compile a due-diligence snapshot for vendor 'cipla'. I need: founding year, latest funding round (am
[2] AIMessage: tool_calls = ['get_current_date({})', "get_company_profile({'company_name': 'Cipla'})", "get_funding_rounds({'company_name': 'Cipla'})", "search_news({'company_name': 'Cipla', 'topic': 'CEO and leadership'})", "search_news({'topic': 'revenue and ARR', 'company_name': 'Cipla'})", "search_news({'topic': 'GDPR legal risk', 'company_name': 'Cipla'})"]
[3] ToolMessage: id=call_6741205719718166776 | 2026-11-11
[4] ToolMessage: id=call_6741205719718168175 | Company profile retrieved from Tavily API.

Title: Cipla Ltd share price | About
[5] ToolMessage: id=call_6741205719718169574 | Results: title:Cipla - Crunchbase Company Profile & Funding, url:https://www.cru
[6] ToolMessage: id=call_6741205719718166877 | CI